# 3. Arm one: robustness and the directive confound

Reproduces the section *Robustness* (holding out whole applications, the directive confound). Cheap
analyses are refitted; the long ones are read from the frozen result files and checked against the numbers the
report quotes.

In [1]:
from common import *
from scipy.stats import binomtest
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, matthews_corrcoef, normalized_mutual_info_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

PAIRS = load_pairs()
BASE = PAIRS[PAIRS.contrast_type == "base_mut"].reset_index(drop=True)
FEATURES = load(DATA / "redmi9t_feature_contract.json")["feature_columns"]
FROZEN = load(RES / "model_results.json")
REPO = load(RES / "repo_level_results.json")
RESTRICTED = load(RES / "confound_restricted_results.json")
REVIEW = load(RES / "posthoc_sensitivity_checks.json")
y = BASE.label.to_numpy()
NAMES = dict(zip([f"mutation_{i}" for i in range(1, 9)], ["nest_deeper", "all_const", "drop_const", "eager_list",
             "extract_widgets", "inline_helpers", "add_siblings", "layout_builder"]))

def logistic(**kw):
    p = dict(class_weight="balanced", penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)
    p.update(kw)
    return Pipeline([("variance", VarianceThreshold()), ("scale", RobustScaler()), ("model", LogisticRegression(**p))])

## Holding out whole applications

Of the held-out contrasts of the seed-group folds, 97.7% come from a repository that is also in training. Only the
grouping unit changes below. The leave-one-project-out scheme is refitted (tuning inside each outer fold, both
loops grouped on the repository); the five-fold repository scheme is read from the frozen file.

In [2]:
proj = BASE.project.to_numpy()
oof = np.full(len(BASE), np.nan)
for i, (tr, te) in enumerate(LeaveOneGroupOut().split(BASE, y, proj)):
    inner = StratifiedGroupKFold(3, shuffle=True, random_state=SEED + 100 + i)
    m = GridSearchCV(logistic(), {"model__C": [0.03, 0.3, 3.0, 30.0]}, scoring="average_precision", cv=inner,
                     refit=True, error_score="raise", n_jobs=1).fit(BASE.loc[tr, FEATURES], y[tr], groups=proj[tr]).best_estimator_
    oof[te] = m.predict_proba(BASE.loc[te, FEATURES])[:, 1]
lopro = roc_auc_score(y, oof)
check("leave-one-project-out AUC", lopro, REPO["leave_one_project_out"]["pooled_auc"], 1e-4)
check("leave-one-project-out AUC (reported)", lopro, 0.784, 5e-4)

sch = REPO["schemes"]
rows = {k: {"AUC": v["intervals"]["roc_auc"]["point"], "low": v["intervals"]["roc_auc"]["ci_low"],
            "high": v["intervals"]["roc_auc"]["ci_high"], "MCC": v["intervals"]["mcc"]["point"]} for k, v in sch.items()}
print(pd.DataFrame(rows).T.round(3).to_string())
check("seed-group AUC", rows["seed_group_5fold"]["AUC"], 0.777, 5e-4)
check("repository five-fold AUC", rows["repository_5fold"]["AUC"], 0.785, 5e-4)
check("repository five-fold CI low", rows["repository_5fold"]["low"], 0.761, 5e-4)
check("repository five-fold CI high", rows["repository_5fold"]["high"], 0.811, 5e-4)
check("repository five-fold MCC", rows["repository_5fold"]["MCC"], 0.467, 5e-4)
check("leave-one-project-out CI low", rows["leave_one_project_out"]["low"], 0.758, 5e-4)
check("leave-one-project-out CI high", rows["leave_one_project_out"]["high"], 0.810, 5e-4)
check("leave-one-project-out MCC", rows["leave_one_project_out"]["MCC"], 0.496, 5e-4)
check("held-out contrasts whose repository is in training", REPO["seed_group_folds_viewed_by_repository"]["mean_test_rows_whose_repo_is_in_train"], 0.977, 5e-4)

lo = REPO["leave_one_project_out"]
auc_by_repo = pd.Series({r: v["roc_auc"] for r, v in lo["per_repository"].items()})
check("repositories above chance", (auc_by_repo > 0.5).sum(), 7)
check("sign test p, 7 of 8", binomtest(7, 8, 0.5, alternative="greater").pvalue, 0.035, 5e-4)
check("weakest repository sits at exactly 0.5", auc_by_repo.min(), 0.5)
check("hold threshold fixed beforehand", REPO["thresholds_fixed_in_advance"]["hold"], 0.72)
print(auc_by_repo.round(3).sort_values().to_string())

                         AUC    low   high    MCC
seed_group_5fold       0.777  0.728  0.828  0.464
repository_5fold       0.785  0.761  0.811  0.467
leave_one_project_out  0.784  0.758  0.810  0.496
amugofjava_anytime_podcast_player              0.500
nknorg_nMobile                                 0.733
SatoshiPortal_bullbitcoin-mobile               0.773
DompetApp_Dompet.flutter                       0.796
deckerst_aves                                  0.816
Hash-Studios_Prism                             0.825
WheretoSleepinNJU_NJU-Class-Shedule-Flutter    0.836
fluttercandies_wechat_flutter                  0.851


## The directive confound

The directive is the identity of the edit (`mutation_1` ... `mutation_8`). Rates, mutual information and the
alias-feature lookup are recomputed from the contrast table.

In [3]:
rate = BASE.groupby("right").label.agg(["size", "sum", "mean"]).rename(columns={"size": "contrasts", "sum": "slower", "mean": "slower share"})
rate.index = [f"{i} ({NAMES[i]})" for i in rate.index]
print(rate.round(3).to_string())
check("lowest slower share (all_const)", rate["slower share"].min(), 0.051, 5e-4)
check("highest slower share (layout_builder)", rate["slower share"].max(), 0.930, 5e-4)
check("normalised mutual information, directive and label", normalized_mutual_info_score(BASE.right, BASE.label), 0.169, 5e-4)
check("smallest directive, contrasts", rate.contrasts.min(), 39)
check("largest directive, contrasts", rate.contrasts.max(), 43)

ALIAS = "d_usesLayoutDependentBuilder"
moves = BASE[BASE[ALIAS] != 0]
check("alias feature non-zero on", len(moves), 42)
check("  all from layout_builder", (moves.right == "mutation_8").all(), 1)
check("  with one sign", moves[ALIAS].gt(0).all(), 1)
check("where the alias moves, 'slower' is right", moves.label.mean(), 0.93, 5e-3)

                              contrasts  slower  slower share
mutation_1 (nest_deeper)             42      23         0.548
mutation_2 (all_const)               39       2         0.051
mutation_3 (drop_const)              43      15         0.349
mutation_4 (eager_list)              42      12         0.286
mutation_5 (extract_widgets)         41       5         0.122
mutation_6 (inline_helpers)          39       5         0.128
mutation_7 (add_siblings)            43      37         0.860
mutation_8 (layout_builder)          43      40         0.930


### Within a directive, and added to the oracle (post-hoc, frozen)

In [4]:
A = REVIEW["A_directive_increment_arm1"]
w = A["within_directive"]
for name, want in {"logreg_balanced": 0.558, "random_forest": 0.550, "rbf_svm": 0.629, "lightgbm": 0.499}.items():
    check(f"size-weighted within-directive AUC, {name}", w[name]["size_weighted_mean_auc"], want, 5e-4)
ci = A["within_directive_bootstrap_95ci"]["rbf_svm"]
check("SVM within-directive interval, low", ci[0], 0.527, 5e-4); check("SVM within-directive interval, high", ci[1], 0.713, 5e-4)
check("directive oracle, directive only", A["incremental"]["directive_only_auc"], 0.843, 5e-4)
check("directive oracle plus the twelve features", A["incremental"]["directive_plus_features_auc"], 0.846, 5e-4)
print("slower / contrasts per directive:", {k: f"{v['slower']}/{v['n']}" for k, v in w["logreg_balanced"]["per_directive"].items()})
check("slower under all_const", w["logreg_balanced"]["per_directive"]["mutation_2"]["slower"], 2)
check("contrasts under all_const", w["logreg_balanced"]["per_directive"]["mutation_2"]["n"], 39)

slower / contrasts per directive: {'mutation_1': '23/42', 'mutation_2': '2/39', 'mutation_3': '15/43', 'mutation_4': '12/42', 'mutation_5': '5/41', 'mutation_6': '5/39', 'mutation_7': '37/43', 'mutation_8': '40/43'}


### Holding out an edit type

Train on seven directives, test on the eighth. The split cannot also be group-disjoint, because every seed group
contributes all eight directives, so the figure is an upper bound on transfer.

In [5]:
oof = np.full(len(BASE), np.nan)
for d in NAMES:
    te = (BASE.right == d).to_numpy()
    oof[te] = logistic(C=1.0).fit(BASE.loc[~te, FEATURES], y[~te]).predict_proba(BASE.loc[te, FEATURES])[:, 1]
check("leave-one-directive-out AUC", roc_auc_score(y, oof), FROZEN["directive_confound"]["leave_one_directive_out"]["pooled"]["roc_auc"], 1e-4)
check("leave-one-directive-out AUC (reported)", roc_auc_score(y, oof), 0.622, 5e-4)
check("leave-one-directive-out MCC", matthews_corrcoef(y, oof >= 0.5), 0.082, 5e-4)
per = FROZEN["directive_confound"]["leave_one_directive_out"]["per_directive"]
above = sum(v["roc_auc"] > 0.5 for v in per.values())
check("directives above chance", above, 6)
check("sign test p, 6 of 8 (two-sided 0.29, one-sided 0.14)", binomtest(6, 8, 0.5, alternative="greater").pvalue, 0.14, 5e-3)

### Removing the strongest carrier

The restricted design drops `layout_builder` and the all-zero-delta rows (frozen result of the restricted run).

In [6]:
fun = RESTRICTED["restriction_funnel"][-1]
check("restricted contrasts", fun["n"], 269)
check("restricted majority baseline", RESTRICTED["majority_baseline"], 0.636, 5e-4)
g = RESTRICTED["oracle_gap"]
check("restricted directive oracle AUC", g["restricted_oracle_auc"], 0.789, 5e-4)
check("restricted logistic regression AUC", RESTRICTED["model_comparison"]["logreg_balanced"]["roc_auc"], 0.707, 5e-4)
check("restricted gap", g["gap_logreg_balanced"], -0.082, 5e-4)
check("full gap", g["full_set_reference"]["gap"], -0.066, 5e-4)
check("gap widened by", abs(g["gap_logreg_balanced"]) - abs(g["full_set_reference"]["gap"]), 0.016, 5e-4)
res_lodir = {d["directive"]: d["roc_auc"] for d in RESTRICTED["leave_one_directive_out"]["per_directive"]}
check("restricted leave-one-directive-out, inline_helpers", res_lodir["mutation_6"], 0.306, 5e-4)

## Figures

The figures for this notebook's sections, written to `figures/` as PNG (300 dpi) and SVG. They plot only values checked above. The images carry no title or note; the caption in the report explains them.

In [7]:
plt = figure_style()
SHORT = lambda p: p.split("_", 1)[1]

# Leave-one-project-out AUC of balanced logistic regression, per held-out application.
n_by_repo = BASE.project.value_counts()
assert set(n_by_repo.index) == set(auc_by_repo.index)
order = auc_by_repo.sort_values()
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(range(len(order)), order, color=MID)
for i, (r, v) in enumerate(order.items()):
    ax.text(0.015, i, f"{v:.3f}  (n={n_by_repo[r]})", va="center", fontsize=7, color="white")
ax.set_yticks(range(len(order)), [SHORT(r) for r in order.index], fontsize=7.5)
ax.axvline(0.5, color=DARK, ls=":", lw=1)
ax.axvline(lopro, color=DARK, ls="--", lw=1)
ax.set_xlim(0, 1.0)
ax.set_xlabel("AUC on the held-out application, balanced logistic regression")
save_figure(fig, "leave_one_project_out")


wrote figures/leave_one_project_out.png and .svg


In [8]:
report()

47 / 47 reported figures reproduced


,claim,value,reported,ok
0,leave-one-project-out AUC,0.783688,0.783688,True
1,leave-one-project-out AUC (reported),0.783688,0.784000,True
2,seed-group AUC,0.777128,0.777000,True
3,repository five-fold AUC,0.784676,0.785000,True
4,repository five-fold CI low,0.761274,0.761000,True
5,repository five-fold CI high,0.811333,0.811000,True
6,repository five-fold MCC,0.466812,0.467000,True
7,leave-one-project-out CI low,0.757938,0.758000,True
8,leave-one-project-out CI high,0.810112,0.810000,True
9,leave-one-project-out MCC,0.495704,0.496000,True
